# Qwen2.5-1.5B Fine-Tuning: Electrical Engineering & Signals and Systems

This Google Colab notebook fine-tunes **Qwen2.5-1.5B-Instruct** on the deduplicated, leak-free **Signals & Systems** dataset (10,467 unique train + 1,179 unique val samples, including multi-sentence derivations and real MathSpeech/NPTEL anchors).

### Pipeline Overview:
1. **Environment & GPU Setup** (Checks for CUDA T4/A100)
2. **Dataset Loading** (Clones repo & decompresses `train.jsonl.gz` / `val.jsonl.gz`)
3. **QLoRA Fine-Tuning** (4-bit NF4, rank 16 LoRA, assistant-only loss masking, `max_length=1024`)
4. **Merge LoRA into Base Model**
5. **GGUF Conversion (Quantized Q4_K_M)** via modern `llama.cpp` CMake build
6. **1-Click Download** of `qwen2.5-1.5b-instruct-q4_k_m.gguf` for Jetson Orin Nano

### Step 1: Install Dependencies & Check GPU

In [ ]:
# Install training dependencies
!pip install -q torch "transformers>=4.40.0" "peft>=0.10.0" "datasets>=2.18.0" "accelerate>=0.28.0" "bitsandbytes>=0.43.0" sentencepiece protobuf gguf

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")
else:
    print("WARNING: No GPU detected! Please go to Runtime > Change runtime type > T4 GPU.")

### Step 2: Download Dataset & Code from Repository
Clones your GitHub repository and verifies the decompressed dataset.

In [ ]:
# Clone the thesis repository (clean clone if re-running)
!rm -rf /content/MTP
!git clone https://github.com/aYush007-blip/MTP.git /content/MTP
%cd /content/MTP/training

# Decompress dataset and inspect line counts
!gzip -d -k -f /content/MTP/training/data/train.jsonl.gz
!gzip -d -k -f /content/MTP/training/data/val.jsonl.gz
!wc -l /content/MTP/training/data/train.jsonl /content/MTP/training/data/val.jsonl
!ls -lh /content/MTP/training/data/

### Step 3: Run Fine-Tuning Script
Trains Qwen2.5-1.5B with 4-bit QLoRA and assistant-only loss masking.

In [ ]:
!python train_qwen_signals.py \
    --model-id "Qwen/Qwen2.5-1.5B-Instruct" \
    --data-dir "/content/MTP/training/data" \
    --output-dir "/content/output_qwen_signals" \
    --epochs 3 \
    --batch-size 4 \
    --grad-accum 4 \
    --max-length 1024 \
    --lr 2e-4 \
    --qlora

### Step 4: Quantize to GGUF (Q4_K_M) for Jetson Orin Nano
Builds `llama-quantize` via CMake and converts the merged HuggingFace model into edge-ready GGUF.

In [ ]:
# Clone and build llama-quantize using CMake
%cd /content
!rm -rf /content/llama.cpp
!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
%cd /content/llama.cpp
!cmake -B build -DGGML_CUDA=OFF
!cmake --build build --target llama-quantize -j$(nproc)

# Convert merged HuggingFace model to FP16 GGUF
!python3 convert_hf_to_gguf.py /content/output_qwen_signals/merged_model --outfile /content/qwen2.5-1.5b-ee-fp16.gguf

# Quantize FP16 GGUF to Q4_K_M (standard high-speed Jetson format)
!./build/bin/llama-quantize /content/qwen2.5-1.5b-ee-fp16.gguf /content/qwen2.5-1.5b-instruct-q4_k_m.gguf Q4_K_M

!ls -lh /content/qwen2.5-1.5b-instruct-q4_k_m.gguf

### Step 5: Download Quantized Model for Jetson
Downloads `qwen2.5-1.5b-instruct-q4_k_m.gguf` (~1.0 GB) to your local machine.
Transfer this file to your Jetson at `/home/ayush02/models/qwen2.5-1.5b-instruct-q4_k_m.gguf`.

In [ ]:
from google.colab import files
print("Downloading quantized GGUF model to your local PC...")
files.download("/content/qwen2.5-1.5b-instruct-q4_k_m.gguf")